# Phase 4B - Forecast Design Validation and Leakage Audit

This notebook validates how the Phase 4 feature models should be interpreted. The earlier feature-model results are promising, but they need to be labelled carefully: lag and rolling demand features are valid for operational one-day-ahead forecasting, while strict multi-step forecasting must recursively generate future lag features.

## Forecast Design Questions

- Are the feature models operational one-day-ahead forecasts?
- Which features are known at forecast time?
- Which features use target history?
- Which same-day exogenous variables may be retrospective unless forecasted or scenario-specified?
- Do stricter recursive feature models still beat the seasonal naive benchmark?

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.forecast_validation import (
    run_forecast_validation,
    validate_lag_feature_design,
    TABLES_DIR,
    MODELLING_FIGURES_DIR,
)

TARGET = 'nd_mean'
STRICT_EXOG_MODE = 'drop'
TEST_START = '2025-01-01'
TEST_END = '2025-12-31'

## Operational One-Day-Ahead Versus Strict Recursive Forecasting

In an operational one-day-ahead setup, each forecast date can use actual demand observed up to the previous day. This makes lag and shifted rolling features valid. In a strict recursive setup, the model cannot use actual target values from later in the test period, so each model's own forecast must be fed back into future lag and rolling features.

In [ ]:
validate_lag_feature_design()

## Run Forecast-Design Validation

The default strict recursive mode drops realised same-day exogenous generation and flow variables. Use `strict_exog_mode='actual'` only as a retrospective/oracle comparison.

In [ ]:
feature_audit, design_comparison, regime_comparison, strict_forecasts = run_forecast_validation(
    target=TARGET,
    test_start=TEST_START,
    test_end=TEST_END,
    strict_exog_mode=STRICT_EXOG_MODE,
)

feature_audit.head(20)

## Feature Availability Audit

Target lag and rolling features are valid for one-day-ahead forecasting when yesterday and earlier demand are known. Calendar features are known for all horizons. Same-day observed renewable generation and interconnector flow variables require forecasts, schedules or scenario assumptions before they can be used prospectively.

In [ ]:
feature_audit.groupby(['feature_type', 'safe_for_one_day_ahead', 'safe_for_strict_multi_step']).size().reset_index(name='feature_count')

## Compare Forecast Designs

This table brings together the existing operational one-day-ahead feature-model results, the strict recursive feature-model results and the seasonal naive benchmark where available.

In [ ]:
design_comparison.sort_values(['forecast_design', 'mape'])

## Strict Recursive Demand-Regime Comparison

The demand-regime table checks whether recursive models remain reliable on high-demand days, where operational planning risk is highest.

In [ ]:
regime_comparison.sort_values(['demand_regime', 'mape'])

## Generated Figures

The validation script saves figures to `outputs/figures/modelling/`.

In [ ]:
from IPython.display import Image, display

for figure_name in [
    'forecast_design_mape_comparison.png',
    'strict_recursive_actual_vs_forecast.png',
    'strict_recursive_regime_mape_comparison.png',
]:
    figure_path = MODELLING_FIGURES_DIR / figure_name
    if figure_path.exists():
        display(Image(filename=str(figure_path)))
    else:
        print(f'Missing figure: {figure_path}')

## Conclusions

- The earlier Phase 4 feature-model results should be described as operational one-day-ahead forecasts.
- The 3.09% MAPE gradient-boosting result is meaningful for next-day forecasting, not a full-year-ahead forecast from the start of 2025.
- Strict recursive evaluation is the appropriate test when future target values cannot be used to construct lag and rolling features.
- If strict recursive models still beat seasonal naive, that strengthens the case for feature-engineered models beyond day-ahead operations.
- If strict recursive results are weaker, the one-day-ahead model remains useful, but longer-horizon deployment needs recursive design, forecasted exogenous inputs or scenario assumptions.